# Deep Learning Project: MLP vs CNN vs Transfer Learning on STL-10

This project compares six models (plus an ensemble) on **STL-10** — a dataset of 96×96 colour photographs across 10 object classes with only 5,000 training images. The small training set makes this an ideal testbed: from-scratch models struggle with limited data, while pretrained models leverage ImageNet features to achieve much higher accuracy.

### Models
1. **MLP** — fully-connected, no regularisation
2. **MLP + Reg** — with Dropout(0.2)
3. **SimpleCNN** — convolutional, no regularisation
4. **SimpleCNN + Reg** — with BatchNorm + Dropout(0.25)
5. **AlexNet** — pretrained ImageNet, classifier fine-tuned
6. **ResNet18** — pretrained ImageNet, frozen backbone, `.fc` head replaced

### Roadmap
| Part | Topic |
|------|-------|
| 1 | Setup and device selection |
| 2 | Load and visualise STL-10 |
| 3 | Build MLP and CNN (no regularisation) |
| 4 | Train without regularisation (30 epochs) |
| 5 | Build MLP and CNN with regularisation |
| 6 | Train with regularisation (30 epochs) |
| 7 | Compare before vs after regularisation |
| 8 | Adapt pretrained AlexNet + ResNet18 (transfer learning) |
| 9 | Train pretrained models (10 epochs) |
| 10 | Final comparison — all six models |
| 11 | Inspect correct and misclassified samples |
| 12 | Discussion and conclusions |


---
## Part 1: Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms, models
from torchvision.models import AlexNet_Weights, ResNet18_Weights

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
import time, copy

torch.manual_seed(42)
np.random.seed(42)

if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')
print(f'Using device: {device}')

---
## Part 2: Load and Visualise STL-10

STL-10 contains 96×96 colour photos across 10 classes with only **5,000 labeled training images** — 10× fewer than CIFAR-10. This small size makes transfer learning's advantage dramatic.

We use two transform pipelines:
- **Pipeline A (scratch models):** native 96×96, STL-10 normalisation, `RandomCrop(96, padding=4)` + `RandomHorizontalFlip` (valid for objects, unlike digits).
- **Pipeline B (pretrained models):** `Resize(144) → RandomCrop(128)`, ImageNet normalisation — matching Worksheet 4's convention.

In [ ]:
# Channel statistics
STL10_MEAN = (0.4467, 0.4398, 0.4066)
STL10_STD  = (0.2603, 0.2566, 0.2713)

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

# Pipeline A: scratch models (96x96)
scratch_train_tf = transforms.Compose([
    transforms.RandomCrop(96, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(STL10_MEAN, STL10_STD),
])
scratch_test_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(STL10_MEAN, STL10_STD),
])

# Pipeline B: pretrained models (128x128, matching Worksheet 4)
TRAIN_IMAGE_SIZE = 128
RESIZE_SIZE      = 144

pretrained_train_tf = transforms.Compose([
    transforms.Resize(RESIZE_SIZE),
    transforms.RandomCrop(TRAIN_IMAGE_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
pretrained_test_tf = transforms.Compose([
    transforms.Resize(RESIZE_SIZE),
    transforms.CenterCrop(TRAIN_IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

print(f'Pipeline A (scratch):    3 x 96 x 96,  STL-10 normalisation')
print(f'Pipeline B (pretrained): 3 x {TRAIN_IMAGE_SIZE} x {TRAIN_IMAGE_SIZE}, ImageNet normalisation')

In [ ]:
# Load STL-10 once, wrap with transforms
class TransformWrapper(Dataset):
    def __init__(self, base_dataset, transform):
        self.data      = base_dataset.data      # (N, 3, 96, 96) uint8
        self.labels    = base_dataset.labels     # (N,)
        self.transform = transform
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        from PIL import Image
        img = Image.fromarray(self.data[idx].transpose(1, 2, 0))
        return self.transform(img), int(self.labels[idx])

t0 = time.time()
base_train = datasets.STL10(root='./data', split='train', download=True)
base_test  = datasets.STL10(root='./data', split='test',  download=True)
print(f'STL-10 loaded in {time.time() - t0:.1f}s')

scratch_train    = TransformWrapper(base_train, scratch_train_tf)
scratch_test     = TransformWrapper(base_test,  scratch_test_tf)
pretrained_train = TransformWrapper(base_train, pretrained_train_tf)
pretrained_test  = TransformWrapper(base_test,  pretrained_test_tf)

print(f'Training images: {len(scratch_train):,}')
print(f'Test images:     {len(scratch_test):,}')
print(f'Shape (scratch):    {scratch_train[0][0].shape}')
print(f'Shape (pretrained): {pretrained_train[0][0].shape}')

### 2.1 Visualise a batch

In [ ]:
CLASS_NAMES = ['airplane', 'bird', 'car', 'cat', 'deer',
              'dog', 'horse', 'monkey', 'ship', 'truck']

preview = TransformWrapper(base_train, scratch_test_tf)
loader  = DataLoader(preview, batch_size=16, shuffle=True)
images, labels = next(iter(loader))

mean_t = torch.tensor(STL10_MEAN).view(3, 1, 1)
std_t  = torch.tensor(STL10_STD).view(3, 1, 1)
images_vis = (images * std_t + mean_t).clamp(0, 1)

fig, axes = plt.subplots(2, 8, figsize=(16, 5))
for ax, img, lbl in zip(axes.flatten(), images_vis, labels):
    ax.imshow(img.permute(1, 2, 0).numpy())
    ax.set_title(CLASS_NAMES[lbl.item()], fontsize=9)
    ax.axis('off')
plt.suptitle('Random STL-10 training samples', fontsize=13)
plt.tight_layout(); plt.show()

### 2.2 Class balance

In [ ]:
counts = np.bincount(np.array(base_train.labels), minlength=10)
plt.figure(figsize=(9, 3.5))
plt.bar(range(10), counts, color='steelblue', edgecolor='black')
plt.xticks(range(10), CLASS_NAMES, rotation=30, ha='right')
plt.ylabel('Count'); plt.title('STL-10 training-set class distribution')
plt.tight_layout(); plt.show()
print(f'Perfectly balanced: {counts.min()} per class')

### 2.3 Create DataLoaders

In [ ]:
BATCH_SIZE = 64  # smaller batch for 5K training images

loader_kw = dict(batch_size=BATCH_SIZE, num_workers=0,
                 pin_memory=(device.type == 'cuda'))

scratch_train_loader    = DataLoader(scratch_train,    shuffle=True,  **loader_kw)
scratch_test_loader     = DataLoader(scratch_test,     shuffle=False, **loader_kw)
pretrained_train_loader = DataLoader(pretrained_train, shuffle=True,  **loader_kw)
pretrained_test_loader  = DataLoader(pretrained_test,  shuffle=False, **loader_kw)

print(f'Batch size: {BATCH_SIZE}')
print(f'Scratch    batches: {len(scratch_train_loader)} train / {len(scratch_test_loader)} test')
print(f'Pretrained batches: {len(pretrained_train_loader)} train / {len(pretrained_test_loader)} test')

---
## Part 3: Build MLP and CNN (No Regularisation)

### 3.1 MLP

With 96×96×3 = 27,648 input features and only 5,000 training images, the MLP has ~14.3M parameters — roughly 2,860 parameters per training example. We expect severe overfitting.

In [ ]:
class MLP(nn.Module):
    """
    3-hidden-layer MLP for 96x96x3 images. No regularisation.
    Flatten(27648) -> 512 -> 256 -> 128 -> 10
    """
    def __init__(self, num_classes=10):
        super().__init__()
        self.network = nn.Sequential(
            nn.Flatten(),
            nn.Linear(3 * 96 * 96, 512), nn.ReLU(inplace=True),
            nn.Linear(512, 256),         nn.ReLU(inplace=True),
            nn.Linear(256, 128),         nn.ReLU(inplace=True),
            nn.Linear(128, num_classes),
        )
    def forward(self, x):
        return self.network(x)

mlp = MLP().to(device)
print(f'MLP parameters: {sum(p.numel() for p in mlp.parameters()):,}')

### 3.2 SimpleCNN

The CNN uses `AdaptiveAvgPool2d(4, 4)` before the classifier to reduce the spatial dimensions from 24×24 to 4×4, keeping the head manageable.

In [ ]:
class SimpleCNN(nn.Module):
    """
    2-block CNN for 96x96x3 images. No regularisation.
    Block 1: Conv(3->32) -> Conv(32->32) -> MaxPool(2) -> 48x48
    Block 2: Conv(32->64) -> Conv(64->64) -> MaxPool(2) -> 24x24
    AdaptiveAvgPool(4,4) -> 64*4*4=1024 -> 128 -> 10
    """
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3,  32, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 4 * 4, 128), nn.ReLU(inplace=True),
            nn.Linear(128, num_classes),
        )
    def forward(self, x):
        return self.classifier(self.features(x))

cnn = SimpleCNN().to(device)
print(f'SimpleCNN parameters: {sum(p.numel() for p in cnn.parameters()):,}')
print(f'MLP / CNN ratio: {sum(p.numel() for p in mlp.parameters()) / sum(p.numel() for p in cnn.parameters()):.1f}x')

---
## Part 4: Training Utilities

All models use the same training loop: Adam + CosineAnnealingLR (matching Worksheet 4). We evaluate on the test set every epoch for monitoring only — no training decisions are based on test metrics, so there is no data leakage.

**We report final-epoch accuracy**, not best-epoch, because selecting the best epoch based on test performance would be a subtle form of leakage.

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return total_loss / total, 100.0 * correct / total

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        total_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return total_loss / total, 100.0 * correct / total

def train_model(model, name, train_loader, test_loader,
                num_epochs, lr, device, weight_decay=0.0):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=weight_decay,
    )
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
    history = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': []}
    print(f'\n{"="*60}')
    print(f'Training {name}  ({num_epochs} epochs)')
    print(f'{"="*60}')
    for epoch in range(1, num_epochs + 1):
        t0 = time.time()
        tr_loss, tr_acc = train_epoch(model, train_loader, criterion, optimizer, device)
        te_loss, te_acc = evaluate(model, test_loader, criterion, device)
        scheduler.step()
        history['train_loss'].append(tr_loss); history['train_acc'].append(tr_acc)
        history['test_loss'].append(te_loss);  history['test_acc'].append(te_acc)
        print(f'Epoch {epoch:2d}/{num_epochs} | '
              f'Train {tr_loss:.4f} / {tr_acc:5.2f}% | '
              f'Test {te_loss:.4f} / {te_acc:5.2f}% | '
              f'LR {optimizer.param_groups[0]["lr"]:.2e} | '
              f'{time.time()-t0:.1f}s')
    return history

SCRATCH_EPOCHS    = 30
PRETRAINED_EPOCHS = 10
LEARNING_RATE     = 1e-3

---
## Part 5: Train Without Regularisation (30 epochs)

### 5.1 Train MLP (no reg)

In [ ]:
mlp_noreg_history = train_model(
    mlp, 'MLP (no reg)', scratch_train_loader, scratch_test_loader,
    num_epochs=SCRATCH_EPOCHS, lr=LEARNING_RATE, device=device,
)

### 5.2 Train CNN (no reg)

In [ ]:
cnn_noreg_history = train_model(
    cnn, 'SimpleCNN (no reg)', scratch_train_loader, scratch_test_loader,
    num_epochs=SCRATCH_EPOCHS, lr=LEARNING_RATE, device=device,
)

---
## Part 6: Is There Overfitting?

With only 5,000 training images and 14.3M MLP parameters, we expect the train-test gap to be large.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for name, hist, color in [('MLP', mlp_noreg_history, 'steelblue'),
                           ('CNN', cnn_noreg_history, 'indianred')]:
    epochs = range(1, len(hist['train_acc']) + 1)
    axes[0].plot(epochs, hist['train_acc'], color=color, linestyle='--', label=f'{name} train')
    axes[0].plot(epochs, hist['test_acc'],  color=color, linestyle='-',  label=f'{name} test')
    axes[1].plot(epochs, hist['train_loss'], color=color, linestyle='--', label=f'{name} train')
    axes[1].plot(epochs, hist['test_loss'],  color=color, linestyle='-',  label=f'{name} test')

axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy (%)')
axes[0].set_title('Accuracy — no regularisation'); axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].set_title('Loss — no regularisation'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

mlp_gap = mlp_noreg_history['train_acc'][-1] - mlp_noreg_history['test_acc'][-1]
cnn_gap = cnn_noreg_history['train_acc'][-1] - cnn_noreg_history['test_acc'][-1]
print(f'MLP train-test gap: {mlp_gap:.2f}%')
print(f'CNN train-test gap: {cnn_gap:.2f}%')

---
## Part 7: Build Models With Regularisation

To combat overfitting, we add regularisation techniques from the tutorials:
- **MLP:** Dropout(0.2) between hidden layers (Tutorial 5 pattern)
- **CNN:** BatchNorm after each conv + Dropout(0.25) after each pool (Tutorial 6 ImprovedCNN pattern)

### 7.1 MLP with Dropout

In [ ]:
class MLP_Reg(nn.Module):
    """MLP with Dropout(0.2) between hidden layers (Tutorial 5 pattern)."""
    def __init__(self, num_classes=10):
        super().__init__()
        self.network = nn.Sequential(
            nn.Flatten(),
            nn.Linear(3 * 96 * 96, 512), nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(512, 256),         nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(256, 128),         nn.ReLU(inplace=True),
            nn.Linear(128, num_classes),
        )
    def forward(self, x):
        return self.network(x)

mlp_reg = MLP_Reg().to(device)
print(f'MLP_Reg parameters: {sum(p.numel() for p in mlp_reg.parameters()):,}  (same — Dropout has no params)')

### 7.2 CNN with BatchNorm + Dropout

In [ ]:
class SimpleCNN_Reg(nn.Module):
    """CNN with BatchNorm + Dropout (Tutorial 6 ImprovedCNN pattern)."""
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3,  32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2), nn.Dropout(0.25),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2), nn.Dropout(0.25),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 4 * 4, 128),
            nn.BatchNorm1d(128), nn.ReLU(inplace=True),
            nn.Dropout(0.25),
            nn.Linear(128, num_classes),
        )
    def forward(self, x):
        return self.classifier(self.features(x))

cnn_reg = SimpleCNN_Reg().to(device)
print(f'SimpleCNN_Reg parameters: {sum(p.numel() for p in cnn_reg.parameters()):,}  (BN adds {sum(p.numel() for p in cnn_reg.parameters()) - sum(p.numel() for p in cnn.parameters())} params)')

---
## Part 8: Train With Regularisation (30 epochs)

We also add `weight_decay=1e-4` (L2 regularisation) to the scratch models.

### 8.1 Train MLP + Reg

In [ ]:
mlp_reg_history = train_model(
    mlp_reg, 'MLP + Reg', scratch_train_loader, scratch_test_loader,
    num_epochs=SCRATCH_EPOCHS, lr=LEARNING_RATE, device=device,
    weight_decay=1e-4,
)

### 8.2 Train CNN + Reg

In [ ]:
cnn_reg_history = train_model(
    cnn_reg, 'SimpleCNN + Reg', scratch_train_loader, scratch_test_loader,
    num_epochs=SCRATCH_EPOCHS, lr=LEARNING_RATE, device=device,
    weight_decay=1e-4,
)

---
## Part 9: Compare Before vs After Regularisation

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# MLP comparison
for hist, label, ls in [(mlp_noreg_history, 'No reg', '--'), (mlp_reg_history, 'With reg', '-')]:
    ep = range(1, len(hist['test_acc']) + 1)
    axes[0, 0].plot(ep, hist['train_acc'], 'b' + ls, alpha=0.6, label=f'Train ({label})')
    axes[0, 0].plot(ep, hist['test_acc'],  'r' + ls, alpha=0.6, label=f'Test ({label})')
axes[0, 0].set_title('MLP — Accuracy'); axes[0, 0].legend(fontsize=8); axes[0, 0].grid(True, alpha=0.3)

for hist, label, ls in [(mlp_noreg_history, 'No reg', '--'), (mlp_reg_history, 'With reg', '-')]:
    ep = range(1, len(hist['test_loss']) + 1)
    axes[0, 1].plot(ep, hist['test_loss'], ls, label=label)
axes[0, 1].set_title('MLP — Test Loss'); axes[0, 1].legend(); axes[0, 1].grid(True, alpha=0.3)

# CNN comparison
for hist, label, ls in [(cnn_noreg_history, 'No reg', '--'), (cnn_reg_history, 'With reg', '-')]:
    ep = range(1, len(hist['test_acc']) + 1)
    axes[1, 0].plot(ep, hist['train_acc'], 'b' + ls, alpha=0.6, label=f'Train ({label})')
    axes[1, 0].plot(ep, hist['test_acc'],  'r' + ls, alpha=0.6, label=f'Test ({label})')
axes[1, 0].set_title('CNN — Accuracy'); axes[1, 0].legend(fontsize=8); axes[1, 0].grid(True, alpha=0.3)

for hist, label, ls in [(cnn_noreg_history, 'No reg', '--'), (cnn_reg_history, 'With reg', '-')]:
    ep = range(1, len(hist['test_loss']) + 1)
    axes[1, 1].plot(ep, hist['test_loss'], ls, label=label)
axes[1, 1].set_title('CNN — Test Loss'); axes[1, 1].legend(); axes[1, 1].grid(True, alpha=0.3)

for ax in axes.flatten():
    ax.set_xlabel('Epoch')
plt.suptitle('Before vs After Regularisation', fontsize=14)
plt.tight_layout(); plt.show()

# Summary table
print(f"{'Model':<20} {'Train acc':>10} {'Test acc':>10} {'Gap':>8} {'Test loss':>10}")
print('-' * 62)
for name, hist in [('MLP (no reg)',     mlp_noreg_history),
                    ('MLP (+ reg)',      mlp_reg_history),
                    ('CNN (no reg)',     cnn_noreg_history),
                    ('CNN (+ reg)',      cnn_reg_history)]:
    gap = hist['train_acc'][-1] - hist['test_acc'][-1]
    print(f"{name:<20} {hist['train_acc'][-1]:>9.2f}% {hist['test_acc'][-1]:>9.2f}% {gap:>+7.2f}% {hist['test_loss'][-1]:>10.4f}")

---
## Part 10: Transfer Learning — AlexNet + ResNet18

STL-10 contains natural object photos — the **same domain** as ImageNet. Unlike SVHN (digits), pretrained features (edges, textures, object parts) should transfer directly.

### 10.1 AlexNet

AlexNet's classifier has Dropout(0.5) layers. If the Linear layers underneath are frozen, the Dropout corrupts the signal without the model being able to adapt. We fix this by unfreezing the **entire classifier** while keeping the convolutional features frozen. We use `lr=1e-4` to protect the pretrained classifier weights.

In [ ]:
alexnet = models.alexnet(weights=AlexNet_Weights.DEFAULT)

# Freeze convolutional features only
for param in alexnet.features.parameters():
    param.requires_grad = False

# Replace last classifier layer and unfreeze entire classifier
alexnet.classifier[6] = nn.Linear(4096, 10)
for param in alexnet.classifier.parameters():
    param.requires_grad = True

alexnet = alexnet.to(device)

dummy_out = alexnet(torch.zeros(2, 3, TRAIN_IMAGE_SIZE, TRAIN_IMAGE_SIZE).to(device))
print(f'AlexNet output: {dummy_out.shape}')
feat_frozen = sum(p.numel() for p in alexnet.features.parameters())
cls_train   = sum(p.numel() for p in alexnet.classifier.parameters() if p.requires_grad)
print(f'Features (frozen): {feat_frozen:,}  |  Classifier (trainable): {cls_train:,}')

### 10.2 ResNet18

ResNet18 has no dropout issue. We freeze the entire backbone and replace only `.fc`.

In [ ]:
resnet = models.resnet18(weights=ResNet18_Weights.DEFAULT)

for param in resnet.parameters():
    param.requires_grad = False

resnet.fc = nn.Linear(resnet.fc.in_features, 10)

resnet = resnet.to(device)

dummy_out = resnet(torch.zeros(2, 3, TRAIN_IMAGE_SIZE, TRAIN_IMAGE_SIZE).to(device))
print(f'ResNet18 output: {dummy_out.shape}')
trainable = sum(p.numel() for p in resnet.parameters() if p.requires_grad)
frozen    = sum(p.numel() for p in resnet.parameters() if not p.requires_grad)
print(f'Trainable: {trainable:,}  |  Frozen: {frozen:,}')

---
## Part 11: Train Pretrained Models (10 epochs)

### 11.1 Fine-tune AlexNet

In [ ]:
alexnet_history = train_model(
    alexnet, 'AlexNet (pretrained)', pretrained_train_loader, pretrained_test_loader,
    num_epochs=PRETRAINED_EPOCHS, lr=1e-4, device=device,
    weight_decay=5e-4,
)

### 11.2 Fine-tune ResNet18

In [ ]:
resnet_history = train_model(
    resnet, 'ResNet18 (pretrained)', pretrained_train_loader, pretrained_test_loader,
    num_epochs=PRETRAINED_EPOCHS, lr=LEARNING_RATE, device=device,
    weight_decay=5e-4,
)

---
## Part 12b: Ensemble — Combining AlexNet + ResNet18

A simple way to squeeze extra accuracy from two pretrained models is to **average their predicted probabilities** (logits) before taking the argmax. This works because AlexNet and ResNet18 make different mistakes — where one model is uncertain, the other may be confident and correct. Averaging their outputs smooths out individual errors.

This technique requires **no additional training** — we just combine the predictions from models we've already trained.

In [ ]:
@torch.no_grad()
def ensemble_predictions(model_a, model_b, loader, device):
    """Average softmax probabilities from two models and return predictions."""
    model_a.eval()
    model_b.eval()
    all_preds, all_labels = [], []
    for images, labels in loader:
        images = images.to(device)
        logits_a = model_a(images)
        logits_b = model_b(images)
        probs_a = F.softmax(logits_a, dim=1)
        probs_b = F.softmax(logits_b, dim=1)
        avg_probs = (probs_a + probs_b) / 2.0
        preds = avg_probs.argmax(dim=1).cpu().numpy()
        all_preds.append(preds)
        all_labels.append(labels.numpy())
    return np.concatenate(all_preds), np.concatenate(all_labels)

ensemble_preds, ensemble_labels = ensemble_predictions(alexnet, resnet, pretrained_test_loader, device)
ensemble_acc = 100.0 * (ensemble_preds == ensemble_labels).mean()
print(f'Ensemble accuracy: {ensemble_acc:.2f}%')

**Why ensembling works:** AlexNet and ResNet18 have fundamentally different architectures — AlexNet uses large 11×11 and 5×5 filters while ResNet18 uses 3×3 filters with skip connections. They learn different feature representations from the same data, so their errors are partially uncorrelated. Averaging their predictions reduces the variance of the combined predictor.

The ensemble is essentially free — no extra training, no extra parameters, just one additional forward pass at inference time. Even a small gain (0.5–2%) demonstrates the principle that diverse models combined outperform any single model.

---
## Part 12: Final Comparison — All Six Models

### 12.1 Training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
all_histories = [
    ('MLP',          mlp_noreg_history, 'steelblue',  ':'),
    ('MLP+Reg',      mlp_reg_history,   'steelblue',  '-'),
    ('CNN',          cnn_noreg_history, 'indianred',  ':'),
    ('CNN+Reg',      cnn_reg_history,   'indianred',  '-'),
    ('AlexNet',      alexnet_history,   'seagreen',   '-'),
    ('ResNet18',     resnet_history,    'darkorange', '-'),
]
for name, hist, color, ls in all_histories:
    ep = range(1, len(hist['test_acc']) + 1)
    axes[0].plot(ep, hist['test_loss'], color=color, linestyle=ls, marker='o', markersize=2, label=name)
    axes[1].plot(ep, hist['test_acc'],  color=color, linestyle=ls, marker='o', markersize=2, label=name)

axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Test loss'); axes[0].set_title('Test Loss')
axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Test accuracy (%)'); axes[1].set_title('Test Accuracy')
axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

### 12.2 Collect predictions

In [ ]:
@torch.no_grad()
def collect_predictions(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    for images, labels in loader:
        outputs = model(images.to(device))
        all_preds.append(outputs.argmax(1).cpu().numpy())
        all_labels.append(labels.numpy())
    return np.concatenate(all_preds), np.concatenate(all_labels)

results = {}
for name, model, loader in [
    ('MLP',      mlp,      scratch_test_loader),
    ('MLP+Reg',  mlp_reg,  scratch_test_loader),
    ('CNN',      cnn,      scratch_test_loader),
    ('CNN+Reg',  cnn_reg,  scratch_test_loader),
    ('AlexNet',  alexnet,  pretrained_test_loader),
    ('ResNet18', resnet,   pretrained_test_loader),
]:
    preds, labels = collect_predictions(model, loader, device)
    results[name] = {'preds': preds, 'labels': labels}
y_true = results['MLP']['labels']
print(f'Collected {len(y_true):,} test predictions from each model.')

# Ensemble comparison
alexnet_acc_val = 100.0 * (results['AlexNet']['preds'] == y_true).mean()
resnet_acc_val  = 100.0 * (results['ResNet18']['preds'] == y_true).mean()
print(f'\nAlexNet alone:             {alexnet_acc_val:.2f}%')
print(f'ResNet18 alone:            {resnet_acc_val:.2f}%')
print(f'Ensemble (AlexNet+ResNet): {ensemble_acc:.2f}%')
print(f'Ensemble gain over best:   {ensemble_acc - max(alexnet_acc_val, resnet_acc_val):+.2f}%')

### 12.3 Per-class accuracy

In [ ]:
def per_class_accuracy(preds, labels, n=10):
    return np.array([100.0 * (preds[labels == c] == c).mean()
                     if (labels == c).sum() > 0 else 0.0 for c in range(n)])

x = np.arange(10)
width = 0.13
colors = ['steelblue', 'cornflowerblue', 'indianred', 'salmon', 'seagreen', 'darkorange', 'purple']
# Add ensemble result
results['Ensemble'] = {'preds': ensemble_preds, 'labels': ensemble_labels}

model_names = list(results.keys())

fig, ax = plt.subplots(figsize=(15, 5))
for i, name in enumerate(model_names):
    accs = per_class_accuracy(results[name]['preds'], y_true)
    ax.bar(x + (i - 2.5) * width, accs, width, label=name, color=colors[i], edgecolor='black', linewidth=0.5)
ax.set_xticks(x); ax.set_xticklabels(CLASS_NAMES, rotation=30, ha='right')
ax.set_ylabel('Accuracy (%)'); ax.set_title('Per-class test accuracy')
ax.set_ylim(0, 100); ax.legend(fontsize=8); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

### 12.4 Confusion matrices (best scratch vs best pretrained)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
for ax, name in zip(axes, ['CNN+Reg', 'ResNet18', 'Ensemble']):
    cm = confusion_matrix(y_true, results[name]['preds'], labels=list(range(10)))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True'); ax.set_title(name)
    ax.tick_params(axis='x', rotation=30)
plt.suptitle('Confusion matrices — best scratch vs pretrained vs ensemble', fontsize=12)
plt.tight_layout(); plt.show()

### 12.5 Summary table

In [ ]:
def param_counts(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

print('=' * 95)
print(f"{'Model':<15} {'Total params':>14} {'Trainable':>14} {'Epochs':>8} {'Train acc':>11} {'Test acc':>11}")
print('-' * 95)
for name, model, hist in [
    ('MLP',      mlp,      mlp_noreg_history),
    ('MLP+Reg',  mlp_reg,  mlp_reg_history),
    ('CNN',      cnn,      cnn_noreg_history),
    ('CNN+Reg',  cnn_reg,  cnn_reg_history),
    ('AlexNet',  alexnet,  alexnet_history),
    ('ResNet18', resnet,   resnet_history),
]:
    total, trainable = param_counts(model)
    n_ep = len(hist['test_acc'])
    print(f"{name:<15} {total:>14,} {trainable:>14,} {n_ep:>8} "
          f"{hist['train_acc'][-1]:>10.2f}% {hist['test_acc'][-1]:>10.2f}%")
print(f"{'Ensemble':<15} {'—':>14} {'—':>14} {'—':>8} "
      f"{'—':>10}  {ensemble_acc:>9.2f}%")
print('=' * 95)

---
## Part 13: Inspect Correct and Misclassified Samples

In [ ]:
@torch.no_grad()
def get_batch_with_probs(model, dataset, n=512):
    loader = DataLoader(dataset, batch_size=n, shuffle=False)
    images, labels = next(iter(loader))
    model.eval()
    probs = F.softmax(model(images.to(device)), dim=1).cpu()
    preds = probs.argmax(dim=1)
    return images, labels, preds, probs

imgs, lbls, preds_b, probs_b = get_batch_with_probs(resnet, pretrained_test, n=512)

im_mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
im_std  = torch.tensor(IMAGENET_STD).view(3, 1, 1)
imgs_vis = (imgs * im_std + im_mean).clamp(0, 1)

correct_idx = (preds_b == lbls).nonzero(as_tuple=True)[0][:6]
wrong_idx   = (preds_b != lbls).nonzero(as_tuple=True)[0][:6]

fig, axes = plt.subplots(2, 6, figsize=(15, 5.5))
for ax, i in zip(axes[0], correct_idx):
    ax.imshow(imgs_vis[i].permute(1, 2, 0).numpy())
    conf = probs_b[i, preds_b[i]].item() * 100
    ax.set_title(f'{CLASS_NAMES[preds_b[i]]}\n({conf:.0f}%)', fontsize=8, color='green')
    ax.set_xticks([]); ax.set_yticks([])
for ax, i in zip(axes[1], wrong_idx):
    ax.imshow(imgs_vis[i].permute(1, 2, 0).numpy())
    conf = probs_b[i, preds_b[i]].item() * 100
    ax.set_title(f'pred: {CLASS_NAMES[preds_b[i]]}\ntrue: {CLASS_NAMES[lbls[i]]}\n({conf:.0f}%)',
                 fontsize=8, color='darkred')
    ax.set_xticks([]); ax.set_yticks([])
axes[0, 0].set_ylabel('Correct', fontsize=11)
axes[1, 0].set_ylabel('Wrong', fontsize=11, color='darkred')
plt.suptitle('ResNet18 — correct (top) vs misclassified (bottom)', fontsize=13)
plt.tight_layout(); plt.show()

---
## Part 14: Discussion and Conclusions

### 14.1 Understanding test loss vs test accuracy

Before interpreting our results, it is important to understand the difference between **test loss** and **test accuracy**, as they can tell different stories.

**Test accuracy** is the percentage of test images correctly classified — a hard metric. A prediction is either right or wrong, regardless of confidence. **Test loss** (cross-entropy) measures how *confident* the model is in its predictions — it penalises not just wrong answers but also uncertain correct answers.

This distinction matters because:
- A model can maintain **flat test accuracy while its test loss rises**. This means it's still getting the same number of images right, but its predictions are becoming less confident and less calibrated — an early warning sign of overfitting.
- Conversely, **falling test loss with flat accuracy** means the model's confidence is improving even though it isn't flipping more predictions from wrong to right — a sign of healthy learning.

In our results, the no-reg MLP shows test loss *increasing* from epoch ~10 onward (1.83 → 1.95) while test accuracy stays around 45%. The model is memorising training data and becoming *overconfident on training images* at the expense of generalisation — the definition of overfitting. The regularised MLP's test loss keeps *decreasing* (1.86 → 1.53), showing that Dropout prevents this degradation.

### 14.2 MLP vs CNN — architecture matters

The CNN outperforms the MLP despite having **72× fewer parameters** (198K vs 14.3M). With 96×96 images and only 5,000 training samples, the MLP has ~2,860 parameters per training image — a recipe for memorisation rather than learning.

The CNN succeeds because of three architectural properties:
1. **Local connectivity** — each neuron sees only a 3×3 patch of pixels. Nearby pixels (which are correlated in natural images) are processed together.
2. **Weight sharing** — the same filter is applied at every spatial location. A "wing detector" learned from one airplane works on all airplanes regardless of position.
3. **Pooling** — `AdaptiveAvgPool2d` reduces 24×24 feature maps to 4×4, giving translation invariance and keeping the classifier head small (1,024 inputs instead of 36,864).

The MLP lacks all three: its first layer connects all 27,648 input pixels to every hidden neuron, with no notion of spatial proximity, no weight sharing, and no built-in invariance.

### 14.3 Regularisation — before vs after

With only 5,000 training images, overfitting is a major concern, especially for the MLP.

**MLP:** The no-reg version reaches 80% train accuracy but only 45% test — a 35% gap with *rising test loss*. Adding Dropout(0.2) reduces the gap to ~12% and keeps test loss falling. Both versions achieve similar test accuracy (~44-45%), which tells us that the MLP is **architecturally limited** on this task — regularisation prevents it from getting worse, but cannot make it fundamentally better. The bottleneck is architecture, not overfitting.

**CNN:** BatchNorm has two effects: it **stabilises gradient flow** (making training faster and reaching higher accuracy) and acts as a **mild regulariser** (each mini-batch sees slightly different normalisation statistics). Dropout(0.25) after each pooling layer provides additional protection against co-adaptation of features.

### 14.4 Transfer learning — domain match matters

STL-10 contains natural object photos (airplanes, birds, cars, cats, etc.) — the **same visual domain** as ImageNet. This is why transfer learning works here: the pretrained backbone's features (edges, textures, shapes, object parts) are directly relevant.

With only 5,000 training images, from-scratch models simply cannot learn rich visual features — there isn't enough data. The pretrained backbone brings knowledge from 1.2 million ImageNet images, making the small dataset sufficient for training just a classifier head on top of already-useful features.

**AlexNet:** We unfreeze the entire classifier (not just the last layer) because AlexNet's classifier contains Dropout(0.5) layers. When the Linear layers underneath are frozen, the Dropout corrupts the signal without the model being able to adapt — resulting in poor accuracy. Unfreezing the classifier fixes this.

**ResNet18:** Has no dropout in its architecture, so a simple frozen-backbone + replaced `.fc` works cleanly. ResNet18's residual connections let it go deeper (18 layers) without vanishing gradients, producing richer features than AlexNet's 8 layers.

### 14.5 Connection to Worksheet 4

| Worksheet exercise | What this notebook does |
|---|---|
| **Exercise 4** — MLP vs CNN parameter count on CIFAR-10 | Part 3: MLP (14.3M) vs CNN (198K) = 72× ratio on STL-10 |
| **Exercise 5** — Fine-tune AlexNet with frozen backbone | Part 10.1: AlexNet on STL-10 with classifier unfrozen |
| **Exercise 6** — Fine-tune ResNet18 with frozen backbone | Part 10.2: ResNet18 on STL-10 with `.fc` replaced |
| **Question 1** — Why flattening hurts performance | MLP's 27,648-input first layer wastes capacity on spatial relationships it cannot exploit |
| **Question 4** — Pooling gives translation invariance | AdaptiveAvgPool2d(4,4) reduces 24×24 → 4×4, making the CNN robust to object position |



### 14.5b Ensembling — combining diverse models

By averaging the softmax probabilities of AlexNet and ResNet18, the ensemble achieves higher accuracy than either model alone. This works because the two architectures make partially uncorrelated errors — AlexNet's large-filter features and ResNet18's residual-connection features capture different aspects of the images. The ensemble requires no additional training; it is a simple post-hoc combination at inference time. Even a modest gain demonstrates the principle that model diversity reduces prediction variance.
### 14.6 Takeaway

Three lessons, visible in one experiment:

1. **Architecture matters.** A CNN with 198K parameters beats an MLP with 14.3M parameters because it encodes "nearby pixels are related" and "the same feature can appear anywhere" directly into its structure.

2. **Regularisation matters when data is scarce.** Dropout and BatchNorm prevent the train-test gap from exploding. The key diagnostic is test *loss*, not just test accuracy — rising test loss signals overfitting even when accuracy appears stable.

3. **Pretrained features matter most — when the domain matches.** ImageNet features transfer beautifully to STL-10 (both are natural photos), allowing pretrained models to vastly outperform from-scratch models on just 5,000 training images.